# Analytics Workshop — Snippets

Copy-pasteable patterns for each step. Replace column names as needed for the specific task cell you are working on. Run these cells in order — each one builds on the variables created in the previous cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

csv_candidates = [
    Path('turbine_telemetry.csv'),
    Path('..') / 'turbine_telemetry.csv',
    Path.cwd() / 'turbine_telemetry.csv',
]
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Run hidden/generate_mock_wind_turbine_telemetry.py first.')

df = pd.read_csv(csv_path, parse_dates=['Timestamp'])
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(f'Turbines: {sorted(df["Turbine_ID"].unique())}')
df.info()
print()
print('Missing values per column:')
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Section 1 — failure state tagging, two-stream split, sort, IQR filter
df['Failure_State'] = np.where(
    df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1).values,
    'Sensor dropout / failure',
    'Normal',
)
failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()
analysis_df = analysis_df.sort_values(['Turbine_ID', 'Timestamp'])

# 3× IQR removes injected impossible spikes; 1.5× would also remove the real degradation signal.
valid_mask = pd.Series(True, index=analysis_df.index)
for col in ['Bearing_Temperature_C', 'Vibration_mm_s']:
    Q1, Q3 = analysis_df[col].quantile(0.25), analysis_df[col].quantile(0.75)
    IQR = Q3 - Q1
    valid_mask &= (analysis_df[col] >= Q1 - 3 * IQR) & (analysis_df[col] <= Q3 + 3 * IQR)
analysis_df = analysis_df[valid_mask]

print(f'Dropout rows kept for forensic review: {len(failure_records)}')
print(f'Rows remaining for analysis: {len(analysis_df)}')

# Sanity check: WT-02 max bearing temp should be higher than WT-01 and WT-03
print(analysis_df.groupby('Turbine_ID')['Bearing_Temperature_C'].describe().round(2))

In [ ]:
# Section 2 — scatter and time-series patterns

# Scatter: bearing temp vs vibration, turbine colour, RPM point size
scatter_df = analysis_df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']).copy()
fig = px.scatter(
    scatter_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
    size='RPM',
    size_max=6,
    hover_data=['Timestamp', 'RPM', 'Wind_Speed_mps', 'Wind_Gust_mps',
                'Turbulence_Intensity', 'Operational_State'],
    title='Vibration vs Bearing Temperature by Turbine',
)
fig.update_layout(template='plotly_white')
fig.show()

# Time series: use raw df filtered to < 500 to preserve gaps and the WT-02 seizure spike
ts_df = df[df['Bearing_Temperature_C'] < 500].copy()
fig2 = px.line(
    ts_df,
    x='Timestamp',
    y='Bearing_Temperature_C',
    color='Turbine_ID',
    title='Bearing Temperature Over Time',
    hover_data=['Operational_State', 'Maintenance_Flag', 'Wind_Speed_mps'],
)
fig2.update_traces(connectgaps=False)
fig2.update_layout(template='plotly_white')
fig2.show()

In [ ]:
# Section 3 — vibration/RPM ratio, rolling mean, WT-01 healthy baseline

wt03_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-03'].sort_values('Timestamp').copy()
wt03_df['Vibration_per_RPM'] = wt03_df['Vibration_mm_s'] / wt03_df['RPM'].replace(0, np.nan)
wt03_df['Vibration_per_RPM_Rolling'] = (
    wt03_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()
)

fig = px.line(
    wt03_df,
    x='Timestamp',
    y='Vibration_per_RPM',
    title='WT-03: Vibration Normalised by RPM (with 4-hour rolling mean)',
    hover_data=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM',
                'Wind_Speed_mps', 'Wind_Gust_mps'],
)
fig.add_scatter(
    x=wt03_df['Timestamp'],
    y=wt03_df['Vibration_per_RPM_Rolling'],
    mode='lines',
    name='4-hour rolling mean',
    line=dict(width=3),
)

# Add WT-01 as a healthy baseline to rule out weather as the explanation
wt01_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-01'].sort_values('Timestamp').copy()
wt01_df['Vibration_per_RPM'] = wt01_df['Vibration_mm_s'] / wt01_df['RPM'].replace(0, np.nan)
wt01_df['Vibration_per_RPM_Rolling'] = (
    wt01_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()
)
fig.add_scatter(
    x=wt01_df['Timestamp'],
    y=wt01_df['Vibration_per_RPM_Rolling'],
    mode='lines',
    name='WT-01 rolling mean (healthy baseline)',
    line=dict(dash='dot', width=2),
)
fig.update_layout(
    template='plotly_white',
    title='Vibration/RPM: WT-03 vs WT-01 healthy baseline',
    xaxis_title='Timestamp',
    yaxis_title='Vibration per RPM',
)
fig.show()

In [ ]:
# Section 4 — warning threshold and linear trend fit

analysis_df['Vibration_per_RPM'] = (
    analysis_df['Vibration_mm_s'] / analysis_df['RPM'].replace(0, np.nan)
)
healthy_baseline = analysis_df.loc[
    analysis_df['Turbine_ID'] == 'WT-01', 'Vibration_per_RPM'
].dropna()
warning_threshold = healthy_baseline.mean() + 2 * healthy_baseline.std()
print(f'WT-01 mean: {healthy_baseline.mean():.4f}')
print(f'WT-01 std:  {healthy_baseline.std():.4f}')
print(f'Warning threshold (mean + 2 SD): {warning_threshold:.4f}')

# Linear trend fit for WT-03 and projected threshold crossing
wt03_ratio = analysis_df.loc[
    analysis_df['Turbine_ID'] == 'WT-03',
    ['Timestamp', 'Vibration_per_RPM', 'Wind_Speed_mps', 'Wind_Gust_mps',
     'Turbulence_Intensity', 'Bearing_Temperature_C'],
].dropna().copy()
wt03_ratio['Day_Index'] = (
    (wt03_ratio['Timestamp'] - wt03_ratio['Timestamp'].min()).dt.total_seconds() / 86400
)
ratio_coeffs = np.polyfit(wt03_ratio['Day_Index'], wt03_ratio['Vibration_per_RPM'], deg=1)
wt03_ratio['Trend_Fit'] = np.polyval(ratio_coeffs, wt03_ratio['Day_Index'])

if ratio_coeffs[0] > 0:
    projected_day = (warning_threshold - ratio_coeffs[1]) / ratio_coeffs[0]
    projected_date = wt03_ratio['Timestamp'].min() + pd.to_timedelta(projected_day, unit='D')
    print(f'Trend slope: {ratio_coeffs[0]:.6f} vibration/RPM per day')
    print(f'Projected threshold crossing: {projected_date.strftime("%Y-%m-%d")}')
else:
    projected_date = None
    print('No positive trend — projection not applicable')

In [ ]:
# Section 4 — summary table (after computing Vibration_per_RPM and running the trend fit)
summary_rows = []
for tid, group in analysis_df.groupby('Turbine_ID'):
    vpr = group['Vibration_per_RPM'].dropna()
    summary_rows.append({
        'Turbine': tid,
        'Mean power (kW)': round(group['Power_kW'].mean(), 1),
        'Max vibration/RPM': round(vpr.max(), 4),
        'Latest Maintenance_Flag': group['Maintenance_Flag'].iloc[-1],
        'Status': group['Operational_State'].iloc[-1],
    })
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))